### Scraping topic ids and total pages

In [ ]:
# import requests
# from bs4 import BeautifulSoup
# import json
# import time

# def scrape_ids_total_pages(topic_id, start=0, step=15):
#     params = {
#         "t": topic_id,
#         "start": start
#     }
#     BASE_TOPIC_URL = "https://neuron.yale.edu/forum/viewtopic.php"
#     HEADERS = {"User-Agent": "Mozilla/5.0"}
#     response = requests.get(BASE_TOPIC_URL, params=params, headers=HEADERS)
#     try:
#         response.raise_for_status()
#     except requests.exceptions.HTTPError as e:
#         if response.status_code == 404:
#             print(f"404 Client Error: Not Found for topic_id: {topic_id}")
#             return []
#         else:
#             raise
#     soup = BeautifulSoup(response.text, 'html.parser')
#     pagination = soup.find("div", class_="pagination")
#     total_page_dict = []
#     if not pagination:
#         total_pages = 1
#     else:
#         page_links = pagination.find_all("a")
#         page_numbers = []
#         for link in page_links:
#             try:
#                 num = int(link.get_text())
#                 page_numbers.append(num)
#             except ValueError:
#                 continue
#         total_pages = max(page_numbers) if page_numbers else 1
#     total_page_dict.append({
#         'topic_id': topic_id,
#         'total_pages': total_pages
#     })

#     return total_page_dict


In [ ]:
# results = []
# for topic_id in range(1, 4749):
#     result = scrape_ids_total_pages(topic_id, start=0, step=15)
#     results.extend(result)
# with open("total_page_dict.json", "w") as f:
#     f.write(json.dumps(results, indent=2))


### Saving posts

In [ ]:
# def save_posts(topic_id, total_pages, start=0, step=15):
#     params = {
#         "t": topic_id,
#         "start": start
#     }
#     BASE_TOPIC_URL = "https://neuron.yale.edu/forum/viewtopic.php"
#     HEADERS = {"User-Agent": "Mozilla/5.0"}
#     response = requests.get(BASE_TOPIC_URL, params=params, headers=HEADERS)
#     posts = []
#     for page in range(total_pages):
#         params = {
#             "t": topic_id,
#             "start": page * step
#         }
#         response = requests.get(BASE_TOPIC_URL, params=params, headers=HEADERS)
#         try:
#             response.raise_for_status()
#         except requests.exceptions.HTTPError as e:
#             if response.status_code == 404:
#                 print(f"404 Client Error: Not Found for url: {response.url}")
#                 break
#             else:
#                 raise
#         soup = BeautifulSoup(response.text, 'html.parser')
#         post_bodies = soup.find_all("div", class_="postbody")
#         for post in post_bodies:
#             content = post.get_text(strip=True, separator='\n')
#             posts.append(content)
#     time.sleep(0.5)
#     return posts

In [ ]:
# with open('total_page_dict.json', 'r') as f:
#     data = json.load(f)

# for item in data:
#     topic_id = item['topic_id']
#     total_pages = item['total_pages']
#     groups = [data[i:i+100] for i in range(0, len(data), 100)]
# for i, group in enumerate(groups):
#     for item in group:
#         topic_id = item['topic_id']
#         total_pages = item['total_pages']
#         posts = save_posts(topic_id, total_pages)
#         if posts:  # Only write if posts is not empty
#             file_path = f"/Users/riesakai/Desktop/MCDOUGAL_LAB/neuron_forum/neuron_forum_posts/scraped_threads/threads_{i}.txt"
#             with open(file_path, "a", encoding="utf-8") as f:
#                 f.write(f"TOPIC_ID: {topic_id}\n")
#                 for post in posts:
#                     f.write(post)
#                     f.write(f"\n--------------------\n")
#                 f.write("="*40 + "\n")
#     print(f"Result Saved for Group {i}")

### Extract posts from files (list of strings) & topic_ids and dates (list of dictionaries)

In [1]:
import os
from datetime import datetime
import re

def extract_titles_from_files(directory):
    titles = []
    post_chunks = []
    date_pattern = re.compile(r"^[A-Za-z]{3} [A-Za-z]{3} \d{2}, \d{4} \d{1,2}:\d{2} [ap]m$")

    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if filepath.endswith('.txt'):
            with open(filepath, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                i = 0
                current_chunk = []
                while i < len(lines):
                    line = lines[i]
                    current_chunk.append(line)
                    if line.strip() == "========================================":
                        # Extract topic_id, title, and last date from current_chunk
                        chunk_lines = [l.strip('\n') for l in current_chunk]
                        topic_id = None
                        title = None
                        last_date_str = None
                        first_date_str = None
                        for idx, l in enumerate(chunk_lines):
                            if l.startswith("TOPIC_ID:"):
                                topic_id = l.split(":", 1)[1].strip().split()[0]
                                if idx + 1 < len(chunk_lines):
                                    title = chunk_lines[idx + 1].strip()
                            if date_pattern.match(l):
                                if first_date_str is None:
                                    first_date_str = l
                                    dt_first = datetime.strptime(first_date_str, "%a %b %d, %Y %I:%M %p")
                                    date_only_first = dt_first.date()
                                last_date_str = l
                        if topic_id and title and last_date_str:
                            dt_last = datetime.strptime(last_date_str, "%a %b %d, %Y %I:%M %p")
                            date_only_last = dt_last.date()
                            
                        else:
                            date_only_last = None
                            date_only_first = None
                        titles.append({'id': int(topic_id) if topic_id else None,
                                       'topic': title,
                                       'first_date': date_only_first,
                                       'last_date': date_only_last
                                       })
                        # Save the full post chunk
                        post_chunks.append("".join(current_chunk))
                        current_chunk = []
                    i += 1
    return titles, post_chunks

# Usage:
titles, post_chunks = extract_titles_from_files("../../neuron_forum/neuron_forum_posts/scraped_threads")
sublist = [{'id': d['id'], 'title': d['topic']} for d in titles]

In [6]:
# All posts
post_chunks[:5]
#print(len(post_chunks))

['TOPIC_ID: 1533\nCellbuilder hangs the system\nQuote\nPost\nby\nbadjoby\n»\nSun Mar 22, 2009 5:56 pm\nHi,\nUsing NEURON -- Release 7.0 (281:80827e3cd201) 2009-01-16\non Ubuntu 8.10 and 9.04\n1) start nrngui\n2) Build->Cell builder\n3) The check boxes are not checkable including Cont Create.\n4) Attempt to click on the check boxes 2-3 times cause the system hang.\nbut If I do the following\n1) start nrngui\n2) At oc> prompt enter the following\nobjectvar ocbox__\n{\nload_file("celbild.hoc", "CellBuild")\n}\n{ocbox_ = new CellBuild(1)}\n{\nocbox_.map("CellBuild[0]", 117, 494, 788.16, 358.08)\n}\n(from your program)\nthen Cell builder comes up and the everything including check boxes are fine.\nthanks, joby\n--------------------\nRe: Cellbuilder hangs the system\nQuote\nPost\nby\nted\n»\nMon Mar 23, 2009 10:24 am\nDid you install with the rpm or did you get the gzipped tar files and compile the source code?\nDoes neurondemo work?\n--------------------\nRe: Cellbuilder hangs the system\nQ

In [2]:
topic_id_from_files = [id_dict['id'] for id_dict in titles]

### Removing Names

In [3]:
# didn't use this one
def clean_post(text):
    cleaned_lines = []
    lines = text.splitlines()
    i = 0
    while i < len(lines):
        line = lines[i]
        # Skip lines containing "Quote", "Post", or "by"
        if line.strip() in {"Quote", "Post", "by"}:
            i += 1
            continue
        # If line is "»", skip it and also skip the previous line
        if line.strip() == '»' and i > 0:
            cleaned_lines.pop()  # Remove the last added line (the one above "»")
            i += 1
            continue
        cleaned_lines.append(line)
        i += 1
    return "\n".join(cleaned_lines)

In [ ]:
# import spacy
# from tqdm import tqdm

# def remove_names(thread):
#     nlp = spacy.load("en_core_web_trf")
#     doc = nlp(thread)
#     new_text = thread
#     for ent in doc.ents:
#         if ent.label_ == "PERSON":
#             new_text = new_text.replace(ent.text, "---")
#     return new_text

# post_chunks_no_names = []
# for post in tqdm(post_chunks, desc="Removing names"):
#     new_post = remove_names(post)
#     post_chunks_no_names.append(new_post)


In [4]:
new_post_chunks = []
for post in post_chunks:
    new_post = clean_post(post)
    new_post_chunks.append(new_post)

In [ ]:
# with open("post_chunks_no_names.txt", "w", encoding="utf-8") as f:
#     for post in post_chunks_no_names:
#         f.write(post)
#         f.write("\n" + "="*40 + "\n")

In [ ]:
# topic_ids = []
# with open("post_chunks_no_names.txt", "r", encoding="utf-8") as f:
#     lines = f.readlines()
#     for line in lines:
#         if line.startswith("TOPIC_ID: "):
#             topic_id = line.split(":", 1)[1].strip().split()[0]
#             topic_ids.append(topic_id)
     

In [ ]:
# matches = []
# for i, s in enumerate(posts_in_14):
#     if "good question" in s.lower():
#         lines = s.splitlines()
#         topic_id = None
#         title = None
#         if lines:
#             if lines[0].startswith("TOPIC_ID:"):
#                 try:
#                     topic_id = int(lines[0].split(":", 1)[1].strip().split()[0])
#                 except Exception:
#                     topic_id = None
#             if len(lines) > 1:
#                 title = lines[1].strip()
#         matches.append({
#             "index": i,
#             "topic_id": topic_id,
#             "title": title,
#             "snippet": s  # first 200 chars as a preview
#         })
# topic_ids = [m.get("topic_id") for m in matches if m.get("topic_id") is not None]


### Prompt

In [ ]:
import openai

def score_threads(thread):
    
    client = openai.OpenAI(
        organization="org-3z6NAgNdNa6W5HskVBdfqbxJ"
    )

    prompt = (

    f"""
    The following is a thread from the NEURON simulator's forum. 
    Threads that receive high scores are considered candidates for inclusion in the NEURON documentation.
    For this thread, rate the following 5 criteria on a scale from 3 (highest) to 0 (lowest). 
    Do not use 3 unless the thread strongly aligns with the criterion. 

    {thread}

    Criteria:
    1. How much of this thread is still valid and relevant for current documentation? Score 0 if the information in this thread is outdated.
    2. How much does this thread relate to the functionality of NEURON? Score low if the thread is more about other tools or specific models.
    3. How much of this thread offer insights that have not been mentioned previously? Score 0 if the answer only includes links to other threads or existing documentation.  
    4. How applicable is this thread to the general NEURON users? Score low if the discussion focuses on a highly specific issue. Score high if other NEURON users would also be interested in this topic.
    5. How well is the post received by the people who answered? Score high if the tone of the answers is positive. Score 0 if there are no replies. 
    """ +
    """Provide your final answer in the following format. TOPIC_ID is the number from the thread and each field corresponds to one of the questions and ## represents the corresponding score: {"TOPIC_ID": {"validity": ##, "direct_NEURON": ##, "insights": ##, "broad_applicability": ##, "well_received": ##}},. Provide no other output after this concluding JSON."""
    )

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    scored = response.choices[0].message.content

    return scored

**Examples**
for example, announcements for a NEURON course in 2024 would be scored 0

for example, posts about events unrelated to NEURON would receive a score of 0

for example, answers that provide step-by-step solution would be scored 3

for example, someone asking where to find information for ion channel distribution for neuron types would be of general interest and score a 3 while someone asking about something tied closely to specific library versions and unusual hardware architecture would be scored 0

In [ ]:
# # final version
# from tqdm import tqdm
# import openai
# import time

# for post in tqdm(new_post_chunks, desc="Scoring threads"):
#     scored = score_threads(post)
#     with open("scored_threads_final-3.json", 'a', encoding='utf-8') as f:
#         f.write(f"\n{scored}")

### Test with 100 to evaluate consistency

In [ ]:
with open('scored_threads-38.json', 'r', encoding='utf-8') as f:
    lines = f.readlines()
test_100 = []
for line in lines:
    if line.strip().startswith('{') and ':' in line:
        topic_id = line.split('"', 2)[1]
        test_100.append(int(topic_id))

test_100

In [ ]:
test_posts = []
for post in new_post_chunks:
        lines = post.splitlines()
        if lines and lines[0].startswith("TOPIC_ID:"):
            topic_id = lines[0].split(":", 1)[1].strip().split()[0]
            if int(topic_id) in test_100:
                test_posts.append(post)

In [ ]:
# first rater
# for post in test_posts:
#     scored = score_threads(post)
#     with open("scored_threads_o.json", 'a', encoding='utf-8') as f:
#             f.write(f"\n{scored}")

In [ ]:
# second rater
# for post in test_posts:
#     scored = score_threads(post)
#     with open("scored_threads_p.json", 'a', encoding='utf-8') as f:
#             f.write(f"\n{scored}")

In [ ]:
# Convert into dfs

import pandas as pd
import json

def json_df(filepath):
# Read the JSON files as raw data first
    with open(filepath, 'r') as f:
        data = json.load(f)
# Flatten the structure for data_1
    flat = []
    for entry in data:
        for k, v in entry.items():
            v['id'] = int(k)  # move key into a field
            flat.append(v)
    # Convert to DataFrames
    df = pd.DataFrame(flat)
    df_sorted = df.sort_values(by='id')
    return df_sorted

df_1 = json_df('scored_threads_m.json')
df_2 = json_df('scored_threads_n.json')

### 1. QWK

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Calculate kappa for each scoring dimension
for col in df_1.columns[:5]:
        qwk = cohen_kappa_score(df_1[col], df_2[col], weights='quadratic')
        print(f"Quadratic Weighted Kappa for {col}: {qwk:.2f}")

### 2. Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


merged = pd.merge(df_1, df_2, on="id", suffixes=("_1", "_2"))
# Melt both dataframes to long format
df1_long = df_1.melt(id_vars='id', var_name='feature', value_name='rater1')
df2_long = df_2.melt(id_vars='id', var_name='feature', value_name='rater2')


In [ ]:
diff = pd.DataFrame()
diff['id'] = merged['id']
cols = df_1.columns

for col in cols[:5]:
    diff[f'{col}_diff'] = abs(merged[f'{col}_1'] - merged[f'{col}_2'])

diff.set_index('id', inplace=True)

# Plot the heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(diff, cmap='coolwarm', linewidths=0.5)
plt.title("Heatmap of Rating Differences (df1 vs df2)")
plt.ylabel("TOPIC ID")
#plt.xlabel("Feature")
plt.show()

In [ ]:
# Calculate disagreement rate for each feature
disagreement_rates = {}
for feature in cols[:-1]:  # exclude 'id'
    disagreements = (merged[f"{feature}_1"] != merged[f"{feature}_2"]).sum()
    total = len(merged)
    disagreement_rates[feature] = disagreements / total

print("Disagreement rates per feature:")
for feature, rate in disagreement_rates.items():
    print(f"{feature}: {rate:.2%}")

In [ ]:
# Agreement rate: proportion of cases where rater1 == rater2 for all features
# Calculate agreement counts for all features except 'id'
agreement_counts = 0
for col in cols[:-1]:  # exclude 'id'
    agreement_counts += (merged[f"{col}_1"] == merged[f"{col}_2"]).sum()
total_comparisons = merged.shape[0] * (len(cols) - 1)
agreement_rate = agreement_counts / total_comparisons
print(f"Overall agreement rate: {agreement_rate:.2%}")

In [ ]:
agreement_rates = {}
for feature in cols[:-1]:  # exclude 'id'
    agreements = (merged[f"{feature}_1"] == merged[f"{feature}_2"]).sum()
    total = len(merged)
    agreement_rates[feature] = agreements / total

print("Agreement rates per criterion:")
for feature, rate in agreement_rates.items():
    print(f"{feature}: {rate:.2%}")

In [ ]:
import numpy as np

# Remap features using merged_long instead of melt_merged
merged_long = pd.merge(df1_long, df2_long, on=['id', 'feature'])
mapping = {'validity':'Validity', 'direct_NEURON': 'Relevance to NEURON', 'insights': 'Insights', 'broad_applicability': 'Broad Applicability', 'well_received': 'Community Response'}
merged_long['feature'] = merged_long['feature'].map(mapping)
features = merged_long['feature'].unique()

percent_matrices = []
for feature in features:
    df = merged_long[merged_long['feature'] == feature].copy()
    df = df[['rater1', 'rater2']]
    df['rater1'] = df['rater1'].clip(0, 3).astype(int)
    df['rater2'] = df['rater2'].clip(0, 3).astype(int)
    region_counts = pd.crosstab(df['rater1'], df['rater2'])
    region_percents = region_counts / region_counts.values.sum() * 100
    # Sort index descending so 3 is at the top, 0 at the bottom
    region_percents = region_percents.sort_index(ascending=False)
    percent_matrices.append(region_percents)

all_values = np.concatenate([df.values.flatten() for df in percent_matrices if not df.empty])
all_values = np.concatenate([df.values.flatten() for df in percent_matrices])

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=False, sharey=False)
axes = axes.flatten()
vmin, vmax = np.nanmin(all_values), np.nanmax(all_values)

for i, (feature, region_percents) in enumerate(zip(features, percent_matrices)):
    sns.heatmap(region_percents, annot=True, fmt=".1f", cmap="YlGnBu", vmin=vmin, vmax=vmax, ax=axes[i], cbar=False,
                yticklabels=True)
    axes[i].set_title(feature)
    axes[i].set_xlabel("Second Run")
    axes[i].set_ylabel("First Run")

if len(features) < len(axes):
    for j in range(len(features), len(axes)):
        axes[j].axis('off')

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
sns.heatmap(percent_matrices[0], cmap="YlGnBu", vmin=0, vmax=100, cbar=True, cbar_ax=cbar_ax, annot=False)
cbar_ax.clear()
fig.colorbar(
    plt.cm.ScalarMappable(cmap="YlGnBu", norm=plt.Normalize(vmin, vmax)),
    cax=cbar_ax,
    label="Percent"
)
fig.suptitle("Agreement between two raters", fontsize=20, y=1)

fig.text(0.78, 0.32, """Agreement rates per criterion:\n
Validity: 89.00%\n
Relevance to NEURON: 87.00%\n
Insights: 92.00%\n
Broad Applicability: 89.00%\n
Community Response: 80.00%""",
         ha='center', va='center', fontsize=14)

plt.show()


In [ ]:
import numpy as np

# Remap features using merged_long instead of melt_merged
merged_long = pd.merge(df1_long, df2_long, on=['id', 'feature'])
mapping = {'validity':'Validity', 'direct_NEURON': 'Relevance to NEURON', 'insights': 'Insights', 'broad_applicability': 'Broad Applicability', 'well_received': 'Community Response'}
merged_long['feature'] = merged_long['feature'].map(mapping)
features = merged_long['feature'].unique()

percent_matrices = []
for feature in features:
    df = merged_long[merged_long['feature'] == feature].copy()
    df = df[['rater1', 'rater2']]
    df['rater1'] = df['rater1'].clip(0, 3).astype(int)
    df['rater2'] = df['rater2'].clip(0, 3).astype(int)
    region_counts = pd.crosstab(df['rater1'], df['rater2'])
    region_percents = region_counts / region_counts.values.sum() * 100
    # Sort index descending so 3 is at the top, 0 at the bottom
    region_percents = region_percents.sort_index(ascending=False)
    percent_matrices.append(region_percents)

all_values = np.concatenate([df.values.flatten() for df in percent_matrices if not df.empty])
all_values = np.concatenate([df.values.flatten() for df in percent_matrices])

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=False, sharey=False)
axes = axes.flatten()
vmin, vmax = np.nanmin(all_values), np.nanmax(all_values)

for i, (feature, region_percents) in enumerate(zip(features, percent_matrices)):
    sns.heatmap(region_percents, annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100, ax=axes[i], cbar=False,
                yticklabels=True)
    axes[i].set_title(feature)
    axes[i].set_xlabel("Second Run")
    axes[i].set_ylabel("First Run")

if len(features) < len(axes):
    for j in range(len(features), len(axes)):
        axes[j].axis('off')

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar_ax.clear()
fig.colorbar(
    plt.cm.ScalarMappable(cmap="YlGnBu", norm=plt.Normalize(vmin=0, vmax=100)),
    cax=cbar_ax,
    label="Percent"
)
fig.suptitle("Agreement between two raters", fontsize=20, y=1)

fig.text(0.78, 0.32, """Agreement rates per criterion:\n
Validity: 89.00%\n
Relevance to NEURON: 87.00%\n
Insights: 92.00%\n
Broad Applicability: 89.00%\n
Community Response: 80.00%""",
         ha='center', va='center', fontsize=14)

plt.show()


In [ ]:
df_1.head()

In [ ]:
# Calculate average scores for rater 1 and rater 2 for each row
df_1['total_score'] = df_1[['validity', 'direct_NEURON', 'insights', 'broad_applicability', 'well_received']].sum(axis=1)
df_2['total_score'] = df_2[['validity', 'direct_NEURON', 'insights', 'broad_applicability', 'well_received']].sum(axis=1)

# Merge average scores by 'id'
combined_avg_scores = pd.DataFrame({
    'id': df_1['id'],
    'rater1_avg': df_1['total_score'],
    'rater2_avg': df_2['total_score']
})
combined_avg_scores['min_avg'] = combined_avg_scores[['rater1_avg', 'rater2_avg']].mean(axis=1)

# Plot histogram of the combined average score (minimum of both raters)
plt.figure(figsize=(6, 4))
combined_avg_scores['min_avg'].value_counts().sort_index().plot(kind='bar', width=1, edgecolor='black')
plt.xlabel('Minimum Average Score (Rater 1 & Rater 2)')
plt.ylabel('Frequency')
plt.title('Histogram of Minimum Combined Average Scores')
plt.tight_layout()
plt.show()

### Results

### Characteristics of the dataset

In [3]:
import pandas as pd

df_titles = pd.DataFrame(titles)
# Extract year from 'first_date'
df_titles['year'] = df_titles['first_date'].apply(lambda x: x.year if pd.notnull(x) else None)

In [4]:
import pandas as pd

import matplotlib.pyplot as plt

# Extract year from 'first_date'
df_titles['year'] = df_titles['first_date'].apply(lambda x: x.year if pd.notnull(x) else None)

# Count threads per year
year_counts = df_titles['year'].value_counts().sort_index()

# Plot
plt.figure(figsize=(10, 5))
year_counts.plot(kind='bar', edgecolor='black', width=1)
plt.xlabel('Year')
plt.xticks(rotation=0)
plt.ylabel('Number of Threads')
plt.title('Thread Counts by Year (first_date)')
plt.tight_layout()
plt.show()

selected_df = df_titles[df_titles['id'].isin(ids_15 or ids_14)]

# for id in df_titles['id']:
#     if id in ids_14 or id in ids_15:
#         df_selected = df_titles[df_titles['id'].isin(ids_14 + ids_15)]
# plt.figure(figsize=(10, 5))
# df_selected['year'].value_counts().sort_index().plot(kind='bar', edgecolor='black', width=1)
# plt.xlabel('Year')
# plt.xticks(rotation=0)
# plt.ylabel('Number of Selected Threads')
# plt.title('Selected Thread Counts by Year (first_date)')
# plt.tight_layout()
# plt.show()
# # Overlay the selected thread counts on the original year bar plot with aligned x-axis

# plt.figure(figsize=(10, 5))
# all_years = sorted(set(year_counts.index).union(df_selected['year'].value_counts().index))
# all_counts = year_counts.reindex(all_years, fill_value=0)
# selected_counts = df_selected['year'].value_counts().reindex(all_years, fill_value=0)

# plt.bar(all_years, all_counts, edgecolor='black', width=1, color='lightgray', label='All Threads')
# plt.bar(all_years, selected_counts, edgecolor='black', width=1, color='orange', alpha=0.7, label='Selected Threads')
# plt.xlabel('Year')
# plt.xticks(all_years, rotation=0)
# plt.ylabel('Number of Threads')
# plt.title('Thread Counts by Year')
# plt.legend()
# plt.tight_layout()
# plt.show()


NameError: name 'df_titles' is not defined

In [ ]:
from datetime import datetime

threads_with_3plus_people = []
for post in post_chunks:
    lines = post.splitlines()
    authors = set()
    for i, line in enumerate(lines):
        
        # Find author's name: one line above '»'
        if line.strip() == '»' and i > 0:
            author_name = lines[i - 1].strip()
            if author_name:
                authors.add(author_name)
    if len(authors) >= 3:
        # Find topic_id for this post
        topic_id = None
        if lines and lines[0].startswith("TOPIC_ID:"):
            topic_id = int(lines[0].split(":", 1)[1].strip().split()[0])
        # Find title from titles list
        title = next((t['topic'] for t in titles if t['id'] == topic_id), None)
        threads_with_3plus_people.append({'id': topic_id, 'title': title, 'authors': list(authors)})

#len(threads_with_3plus_people) / len(df_titles) * 100
threads_with_3plus_people


In [ ]:
threads_with_2plus_people = []
for post in post_chunks:
    lines = post.splitlines()
    authors = set()
    for i, line in enumerate(lines):        
        # Find author's name: one line above '»'
        if line.strip() == '»' and i > 0:
            author_name = lines[i - 1].strip()
            if author_name:
                authors.add(author_name)
    if len(authors) >= 2:
        # Find topic_id for this post
        topic_id = None
        if lines and lines[0].startswith("TOPIC_ID:"):
            topic_id = int(lines[0].split(":", 1)[1].strip().split()[0])
        # Find title from titles list
        title = next((t['topic'] for t in titles if t['id'] == topic_id), None)
        threads_with_2plus_people.append({'id': topic_id, 'title': title, 'authors': list(authors)})

len(threads_with_2plus_people) / len(df_titles) * 100

In [ ]:
threads_with_one_person = []
for post in post_chunks:
    lines = post.splitlines()
    authors = set()
    for i, line in enumerate(lines):        
        # Find author's name: one line above '»'
        if line.strip() == '»' and i > 0:
            author_name = lines[i - 1].strip()
            if author_name:
                authors.add(author_name)
    if len(authors) == 1:
        # Find topic_id for this post
        topic_id = None
        if lines and lines[0].startswith("TOPIC_ID:"):
            topic_id = int(lines[0].split(":", 1)[1].strip().split()[0])
        # Find title from titles list
        title = next((t['topic'] for t in titles if t['id'] == topic_id), None)
        threads_with_one_person.append({'id': topic_id, 'title': title, 'authors': list(authors)})

len(threads_with_one_person) / len(df_titles) * 100

In [ ]:
no_reply = [thread for thread in threads_with_one_person if thread['authors'][0] not in ['ted', 'ramcdougal', 'hines', 'tom']]
(len(threads_with_one_person) - len(no_reply)) / len(df_titles) * 100

In [ ]:
# Create the figure
plt.figure(figsize=(14, 6))

score_df = df_sorted.drop(columns=['id', 'total_score'])
# Melt the DataFrame to long format for seaborn
df_long = score_df.melt(var_name='Question', value_name='Score')

# Rename the values in the "Question" column
rename_dict = {
    "validity": "Validity",
    "direct_NEURON": "Direct NEURON Use",
    "insights": "Insights",
    "broad_applicability": "Broad Applicability",
    "well_received": "Community Response"
}
df_long["Question"] = df_long["Question"].replace(rename_dict)

### Score Distribution

In [ ]:
import pandas as pd
import json

# Read the JSON files as raw data first
with open('scored_threads_final-3.json', 'r') as f:
    data = json.load(f)
# Flatten the structure for data_1
flat = []
for entry in data:
    for k, v in entry.items():
        v['id'] = int(k)  # move key into a field
        flat.append(v)
# Convert to DataFrames
df = pd.DataFrame(flat)


In [ ]:
df_sorted = df.sort_values(by='id')
import matplotlib.pyplot as plt

df_sorted['total_score'] = df_sorted[['validity', 'direct_NEURON', 'insights', 'broad_applicability', 'well_received']].sum(axis=1)
df_sorted['total_score'].value_counts().sort_index().plot(kind='bar', width=1, edgecolor='black')
plt.title('Total Scores')
plt.xlabel('Total Score')
plt.ylabel('Frequency')
plt.xticks(rotation=0)

In [ ]:
import scipy.stats as stats

n = len(df_sorted['total_score'])
mean = df_sorted['total_score'].mean()
std = df_sorted['total_score'].std(ddof=1)
conf_int = stats.t.interval(0.95, n-1, loc=mean, scale=std/(n**0.5))
print(f"Mean: {mean:.1f}")
print(f"95% CI: ({conf_int[0]:.1f}, {conf_int[1]:.1f})")

In [ ]:
df_long = df_sorted.melt(id_vars='id', var_name='feature')
df_long_plot = df_long[df_long['feature']!='total_score']

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap

# --- Prep data ---
# Count rows per (feature, value)
grouped = (
    df_long_plot
    .groupby(['feature', 'value'], dropna=False)
    .size()
    .reset_index(name='count')
)

# Overall-percent (match your original intent of dividing by the dataset size)
grouped['percent'] = grouped['count'] / 4238 * 100

# Desired order & labels
order = ['validity', 'direct_NEURON', 'insights', 'broad_applicability', 'well_received']
order_labels = [rename_dict[o] for o in order]

# Map features to display labels (fallback to original key if missing)
grouped['feature_label'] = grouped['feature'].map(lambda k: rename_dict.get(k, k))

# Keep only rows in our defined order (in case there are extras)
grouped = grouped[grouped['feature'].isin(order)].copy()

# Create numeric x positions to avoid categorical-tick warnings
label_to_pos = {label: i for i, label in enumerate(order_labels)}
grouped['x'] = grouped['feature_label'].map(label_to_pos)

grouped['size'] = grouped['percent'] * 10 + 50 # 10 is adjustable

# --- Figure & axes ---
fig, ax = plt.subplots(figsize=(8, 5))

# Y-axis (score) limits with small padding
ymin = grouped['value'].min()
ymax = grouped['value'].max()
ax.set_ylim(ymin - 0.5, ymax + 0.5)

# Color normalization fixed to 0–100 for consistency across runs
norm = plt.Normalize(0, 100)

# Scatter
scatter = ax.scatter(
    grouped['x'], grouped['value'],
    s=grouped['size'],
    c=grouped['percent'],
    cmap='YlGnBu',
    edgecolors='black',
    linewidths=0.8,
    norm=norm
)

# Colorbar with label & border
cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Percent (%)')


# Wrap only at spaces, no mid-word breaks
wrapped_labels = [
    '\n'.join(textwrap.wrap(lbl, width=12, break_long_words=False, break_on_hyphens=False))
    for lbl in order_labels
]

ax.set_xticks(range(len(order_labels)))
ax.set_xticklabels(wrapped_labels, rotation=0, ha='center')

# Y axis & title
ax.set_ylabel('Score')
ax.set_yticks(sorted(grouped['value'].unique()))
ax.set_title('Distribution of Scores by Criterion')
# Add percentage text above each dot
for _, row in grouped.iterrows():
    ax.text(
        row['x'], row['value'] + 0.20,  # slight offset above the dot
        f"{row['percent']:.1f}%", 
        ha='center', va='bottom', fontsize=9, color='black'
    )

# Final touches
ax.grid(False)
plt.tight_layout()
plt.show()


In [ ]:
len(df_sorted[df_sorted['total_score']==15])

In [ ]:
len(df_sorted[df_sorted['total_score']==14])

In [ ]:
ids_15 = df_sorted[df_sorted['total_score'] == 15]['id'].tolist()
ids_14 = df_sorted[df_sorted['total_score'] == 14]['id'].tolist()

In [ ]:
list_2 = [1947, 2332, 2663, 2747, 2815, 2962, 3021, 4001, 4033, 4388, 70, 136, 138, 150,162, 180, 203, 245, 287, 346, 353, 361, 524, 579, 585, 589, 648, 697, 755, 773, 875, 1168, 1191, 1243, 1260, 1272, 1436, 1459, 1466, 1522, 1724, 1788, 1839, 1905, 1935, 1976, 2048, 2055, 2108, 2245, 2271, 2304, 2336, 2402, 2459, 2475, 2481, 2483, 2489, 2508, 2619, 2656, 2683, 2708, 2714, 2718, 2742, 2840, 2926, 2992, 3016, 3093, 3112, 3127, 3209, 3260, 3294, 3303, 3418, 3422, 3428, 3446, 3452, 3455, 3482, 3485, 3628, 3693, 3811, 3820, 3835, 3883, 3930, 3935, 4032, 4097, 4105, 4167, 4190, 4193, 4236, 4242, 4264, 4265, 4286, 4298, 4303, 4325, 4364, 4373, 4389, 4409, 4411, 4499, 4561, 4593, 4623, 4628, 4652, 4701, 4718, 4720, 4734]
list_1 = [57, 353, 1795, 1928, 2527, 2663, 2822, 3596, 3622, 4046, 4234, 4388, 4420, 68, 70, 150, 173, 279, 300, 313, 344, 461, 499, 524, 673, 708, 766, 773, 779, 782, 818, 897, 902, 982, 1012, 1042, 1078, 1141, 1220, 1243, 1253, 1412, 1419, 1436, 1511, 1517, 1522, 1528, 1598, 1793, 1800, 1816, 1900, 1913, 1974, 1975, 1976, 2048, 2085, 2108, 2131, 2166, 2233, 2263, 2278, 2332, 2356, 2402, 2423, 2437, 2475, 2494, 2508, 2518, 2549, 2553, 2564, 2574, 2576, 2596, 2602, 2617, 2648, 2664, 2696, 2716, 2719, 2747, 2749, 2753, 2815, 2830, 2850, 2887, 2893, 2936, 2949, 2954, 2973, 3067, 3068, 3077, 3093, 3175, 3184, 3211, 3255, 3417, 3428, 3446, 3455, 3457, 3502, 3541, 3549, 3576, 3619, 3650, 3700, 3705, 3712, 3736, 3750, 3785, 3811, 3820, 3859, 3891, 3902, 3904, 3963, 4001, 4042, 4050, 4051, 4097, 4165, 4167, 4227, 4257, 4261, 4264, 4267, 4273, 4286, 4298, 4313, 4338, 4340, 4364, 4372, 4397, 4402, 4409, 4451, 4601, 4605, 4617, 4621, 4623, 4652, 4653, 4664, 4680, 4682, 4720, 4747]

In [ ]:
common = set(list_1) & set(list_2)
len(common)

In [ ]:
selected_titles = []
for title in titles:
    for item in titles:
        if item['id'] in ids_15 or item['id'] in ids_14:
            selected_titles.append(item)

In [ ]:
year_counts

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

# Extract year from 'first_date'
df_titles['year'] = df_titles['first_date'].apply(lambda x: x.year if pd.notnull(x) else None)
# Count threads per year
year_counts = df_titles['year'].value_counts().sort_index()
df_selected = df_titles[df_titles['id'].isin(ids_14) | df_titles['id'].isin(ids_15)]
selected_year_counts = df_selected['year'].value_counts().sort_index()

# Align years for both series
years = sorted(set(year_counts.index) & set(selected_year_counts.index))
x = year_counts.loc[years].values
y = selected_year_counts.loc[years].values

plt.figure(figsize=(10, 6))

# Annotate each point with its year
for xi, yi, year in zip(x, y, years):
    plt.text(xi, yi, str(year), fontsize=8, ha='center', va='center', color='blue')


plt.xlabel('All Threads per Year')
plt.ylabel('Selected Threads per Year')
plt.title('Selected vs. All Thread Counts by Year')
plt.ylim(0, max(y) + 2)

# Regression line
X = np.array(x).reshape(-1, 1)
reg = LinearRegression().fit(X, y)
y_pred = reg.predict(X)
plt.plot(x, y_pred, color='red', linestyle='-', label='Regression Line')
residuals = y - y_pred
std = np.std(residuals)
plt.fill_between(x, y_pred - std, y_pred + std, color='red', alpha=0.2, label='±1 Std. Dev.')

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df_selected[df_selected['year']==2012].sort_values(by='first_date')

## FAQ

In [ ]:
# def prog_ref(thread):
    
#     client = openai.OpenAI(
#         organization="org-3z6NAgNdNa6W5HskVBdfqbxJ"
#     )

#     prompt = (

#     f"""
#     The following post discusses a specific function in NEURON. Please summarize it in one to two sentences to serve as an addition to the programmer reference in NEURON.
    
#     {thread}
#     """)

#     response = client.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant."},
#             {"role": "user", "content": prompt}
#         ]
#     )

#     converted = response.choices[0].message.content

#     return converted

In [ ]:
# pr_list = [70, 150, 180, 353, 361, 524, 579, 585, 648, 755, 875, 1191, 1243, 1260, 1466, 1788, 2048, 2055, 2108, 2332, 2336, 2475, 2481, 2483, 2619, 2663, 2683, 2815, 3093, 3127, 3294, 3446, 3811, 3883, 3935, 4193, 4373, 4561, 4701]

In [ ]:
# pr_posts = []
# for post in new_post_chunks:
#         lines = post.splitlines()
#         if lines and lines[0].startswith("TOPIC_ID:"):
#             topic_id = lines[0].split(":", 1)[1].strip().split()[0]
#             if int(topic_id) in pr_list:
#                 pr_posts.append(post)

In [ ]:
# from tqdm import tqdm
# import openai
# for post in tqdm(pr_posts):
#      scored = prog_ref(post)
#      with open("prog_ref.json", 'a', encoding='utf-8') as f:
#          f.write(f"\n{scored}")

In [ ]:
# list = [136, 138, 162, 203, 245, 346, 589, 773, 1168, 1272, 1436, 1459, 1522, 1724, 1905, 1935, 1947, 1976, 2245, 2459, 2656, 2708, 2714, 2718, 2742 ,2747, 2840, 2926, 2962, 2992, 3021, 3112, 3209, 3260, 3303, 3418, 3422, 3428, 3452, 3455, 3482, 3485, 3628, 3693, 3820, 3835, 3930, 4001, 4032, 4033, 4097, 4105, 4167, 4190, 4236, 4242, 4264, 4286, 4298, 4303, 4325, 4364, 4388, 4389, 4411, 4499, 4593, 4623, 4628, 4652, 4718, 4720, 4734]

In [ ]:
# import os
# import re
# directory = 'docs/short_ver2/'
# # Use the variable 'list' which contains numbers
# rst_files_with_numbers = [] # Change to your target directory if needed


In [ ]:
# import os
# import shutil

# # --- CONFIGURE THESE ---
# source_dir = "docs/short_ver2/"
# target_dir = "docs/faq_rst/"
# numbers = [345, 678, 910]   # your list of numbers
# # -----------------------

# # make sure destination exists
# os.makedirs(target_dir, exist_ok=True)

# # convert numbers to strings for matching
# numbers_str = set(str(num) for num in list)

# for filename in os.listdir(source_dir):
#     if filename.endswith(".rst"):
#         # split on "_" to extract the number part
#         number_part = filename.split("_")[0]
#         if number_part in numbers_str:
#             src_path = os.path.join(source_dir, filename)
#             dst_path = os.path.join(target_dir, filename)

#             try:
#                 shutil.move(src_path, dst_path)
#                 print(f"Moved: {filename}")
#             except Exception as e:
#                 print(f"Failed to move {filename}: {e}")



In [ ]:
# faq_posts = []
# for post in new_post_chunks:
#         lines = post.splitlines()
#         if lines and lines[0].startswith("TOPIC_ID:"):
#             topic_id = lines[0].split(":", 1)[1].strip().split()[0]
#             if int(topic_id) in list:
#                 faq_posts.append(post)

In [ ]:
# import os

# # Directory containing the .rst files
# faq_dir = "neuron_doc/nrn/docs/guide/faq_rst2/"

# for filename in os.listdir(faq_dir):
#     if filename.endswith(".rst"):
#         # Extract the number before the first dot or underscore
#         number = filename.split(".")[0].split("_")[0]
#         thread_url = f"Original Thread: https://neuron.yale.edu/phpBB/viewtopic.php?t={number}"
#         file_path = os.path.join(faq_dir, filename)
#         # Read the file content
#         with open(file_path, "r", encoding="utf-8") as f:
#             content = f.read()
#         # Append the thread URL at the bottom
#         if thread_url not in content:
#             content += "\n\n" + thread_url + "\n"
#             with open(file_path, "w", encoding="utf-8") as f:
#                 f.write(content)

In [ ]:
# """faq_prompt = Please take the following forum post from ## and create a concise summary that can be added to the Frequently Asked Questions section in ##. Return only the result in reStructuredText (RST) format.

# Begin the result with a specific question as the header.

# Provide the summary in clear, concise language.

# Include example code snippets in both Python and Hoc, using the following directives:

# .. code-block:: python


# and

# .. code-block:: hoc"""

In [7]:
d_lst = [136, 138, 162, 203, 245, 346, 589, 773, 1168, 1272, 1436, 1459, 1522, 1724, 1905, 1935, 1947, 1976, 2245, 2459, 2656, 2708, 2714, 2718, 2742, 2747, 2840, 2926, 2962, 2992, 3021, 3112, 3209, 3260, 3303, 3418, 3422, 3428, 3452, 3455, 3482, 3485, 3628, 3693, 3820, 3835, 3930, 4001, 4032, 4033, 4097, 4105, 4167, 4190, 4236, 4242, 4264, 4286, 4298, 4303, 4325, 4364, 4388, 4389, 4411, 4499, 4593, 4623, 4628, 4652, 4718, 4720, 4734, 2402, 3016, 4265, 4409, 287, 1839, 2271, 2304, 2489, 2508, 150, 353, 579, 755, 2048, 2663, 524, 3093, 697, 70, 180, 361, 585, 648, 875, 1191, 1243, 1260, 1466, 1788, 2055, 2108, 2332, 2336, 2475, 2481, 2483, 2619, 2683, 2815, 3127, 3294, 3446, 3811, 3883, 3935, 4193, 4373, 4561, 4701]

In [ ]:
# d_lst = d_lst[["ID", "topic", "function", "language", "FAQ/Programmar's ref", "function/method/class"]]#[d_lst["FAQ/Programmar's ref"] == "Programmar's Ref"]
# d_lst['function'] = d_lst['function'].str.replace(r'\(\)', '', regex=True)
# d_lst.loc[d_lst["language"] == "both", "language"] = "Hoc and Python"

In [ ]:
# progref_ids = d_lst["ID"].tolist()
# for post in new_post_chunks:
#         lines = post.splitlines()
#         if lines and lines[0].startswith("TOPIC_ID:"):
#             topic_id = lines[0].split(":", 1)[1].strip().split()[0]
#             if int(topic_id) in progref_ids:
#                 d_lst.loc[d_lst["ID"] == int(topic_id), "post"] = post

## Join posts and d_lst on ID

In [10]:
import pandas as pd
import re
# Create a mapping from TOPIC_ID to post text
post_dict = {}
pattern = re.compile(r"^TOPIC_ID:\s*(\d+)", re.MULTILINE)
for post in new_post_chunks:
    match = pattern.match(post)
    if match:
        topic_id = int(match.group(1))
        post_dict[topic_id] = post

# Add a 'post' column to d_lst using the mapping
d_lst_df = pd.DataFrame({'id': d_lst})
d_lst_df['post'] = d_lst_df['id'].map(post_dict)

# Save d_lst_df to JSON
d_lst_df.to_json('d_lst_posts.json', orient='records', indent=2)

In [ ]:
# def prog_ref(thread):

#     client = openai.OpenAI(
#         organization="org-3z6NAgNdNa6W5HskVBdfqbxJ"
#     )

#     prompt = (
#         f"""You are a technical documentation assistant helping integrate community Q&A content into NEURON simulator's official documentation 
#     (nrn.readthedocs.io).

#     Given the following forum Q&A thread:

#     {thread}

#     Please do the following:

#     1. IDENTIFY what new information or clarifications the thread contains 
#     that is missing or underdocumented in the official NEURON docs.

#     2. LOCATE the most appropriate place(s) to add this information

#     3. DRAFT the actual .rst text to be inserted.

#     4. FLAG any content that is ambiguous, outdated, or requires verification by a NEURON maintainer before merging.

#     Only include information that is clearly correct and well-supported 
#     by the thread. Do not invent or extrapolate beyond what is stated."""
#     )

#     response = client.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant."},
#             {"role": "user", "content": prompt}
#         ]
#     )

#     converted = response.choices[0].message.content

#     return converted


## RAG 1. Install dependencies

In [1]:
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai openai

## RAG 2. Configuration

In [ ]:
import os

# Path to your locally cloned NEURON repo's docs folder
DOCS_PATHS = [
    ".."
    "docs/nmodl",     # <-- replace with your local path
    "docs/progref",  # <-- replace with your local path
]
  # <-- replace with your local path

# Where to save the persistent index (so you don't re-index every time)
INDEX_STORE_PATH = "docs/neuron_index"

## RAG 3. Build (or load) the index

In [3]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Configure models
Settings.llm = OpenAI(model="gpt-4o", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

if os.path.exists(INDEX_STORE_PATH):
    print("Loading existing index...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_STORE_PATH)
    index = load_index_from_storage(storage_context)
else:
    print("Building index from docs (this may take a few minutes)...")
    all_documents = []
    for docs_path in DOCS_PATHS:
        docs = SimpleDirectoryReader(
            input_dir=docs_path,
            recursive=True,
            required_exts=[".rst"],
        ).load_data()
        print(f"Loaded {len(docs)} .rst files from {docs_path}")
        all_documents.extend(docs)
    print(f"Total: {len(all_documents)} .rst files")
    index = VectorStoreIndex.from_documents(all_documents)
    index.storage_context.persist(persist_dir=INDEX_STORE_PATH)
    print("Index built and saved.")

Loading existing index...


## RAG 4. Define the prompt template

In [34]:
PROMPT_TEMPLATE = """
You are a technical documentation assistant helping integrate community
Q&A content into the NEURON simulator's official documentation.

Below are the most relevant excerpts from the current NEURON documentation,
each labelled with its source .rst file path:

---------------------
{context_str}
---------------------

Here is a forum Q&A thread that contains information to be integrated:

<thread>
{query_str}
</thread>

Instructions:
- Identify every function, method, or class that the thread adds new information about.
- For each one, produce a separate JSON object with exactly these fields:
    - "header": the function/method/class name (e.g. "NetCon.record", "CVode.event", "fadvance")
    - "rst_file": the .rst source file path from the context above (e.g. "docs/hoc/simctrl/cvode.rst")
    - "new_doc": the new .rst text to be inserted, written to match the style of the existing docs.
                 Include code examples where relevant. Be concise and technical.

Return ONLY a JSON array of these objects, with no preamble or explanation.
Example output format:
[
  {
    "header": "NetCon.record",
    "rst_file": "docs/hoc/modelspec/programmatic/network/netcon.rst",
    "new_doc": ".. note::\n\n   ``record()`` can accept a string ...",
    "flag": null
  },
  {
    "header": "CVode.event",
    "rst_file": "docs/hoc/simctrl/cvode.rst",
    "new_doc": "``CVode.event()`` can also accept a proc ...",
    "flag": null
  }
]

Only include information clearly supported by the thread. Do not extrapolate.
"""

## RAG 5. Run a query for a single forum thread

In [13]:
import json
# Paste your forum thread here
with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    entry = json.load(f)

In [ ]:
import json
from llama_index.core import PromptTemplate

query_engine = index.as_query_engine(
    similarity_top_k=5,
    text_qa_template=PromptTemplate(PROMPT_TEMPLATE),
)

def process_thread(thread_id, thread_text):
    """Query the index with a forum thread and return structured JSON results."""
    # Warn if thread is likely to be large (rough estimate: 1 token ~ 4 chars)
    estimated_tokens = len(thread_text) // 4
    if estimated_tokens > 8000:
        print(f"WARNING: Thread {thread_id} is large (~{estimated_tokens} tokens) and may hit the token limit.")
    try:
        response = query_engine.query(thread_text)
        raw = str(response).strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f"WARNING: {thread_id} — could not parse JSON response.")
            print("Raw response:")
            print(raw)
            return [{"header": "PARSE_ERROR", "rst_file": None, "new_doc": raw, "flag": "JSON parse failed"}]
    except Exception as e:
        error_msg = str(e)
        if "maximum context length" in error_msg or "token" in error_msg.lower():
            print(f"ERROR: {thread_id} exceeded the token limit. (~{estimated_tokens} tokens, {len(thread_text)} chars)")
            print("Tip: truncate this thread or reduce similarity_top_k.")
        else:
            print(f"ERROR: {thread_id} failed with: {error_msg}")
        return [{"header": "TOKEN_ERROR", "rst_file": None, "new_doc": None, "flag": f"Failed: {error_msg}"}]


# entry is a list of dicts, so select one entry (e.g., the first)
forum_id = entry[0]["id"]
forum_thread = entry[0]["post"]

result = process_thread(forum_id, forum_thread)
print(json.dumps(result, indent=2))

[
  {
    "header": "CVode",
    "rst_file": "/Users/riesakai/Desktop/MCDOUGAL_LAB/neuron_doc/nrn/docs/progref/simctrl/cvode.rst",
    "new_doc": ".. note::\n\n   NEURON's default integration method is implicit (\"backward\") Euler. This method allows the states of a passive system to reach their \"infinite time\" values in a single integration step when dt is much larger than the largest time constant. This is in contrast to the explicit (\"forward\") Euler method, which NEURON does not use by default.",
    "flag": null
  }
]


## RAG 6. Bulk processing

In [ ]:
# Collect all entries from all threads into a single flat list
all_results = []
for item in entry:
    thread_id = item['id']
    thread_text = item['post']
    print(f"Processing thread {thread_id}...")
    entries = process_thread(thread_id, thread_text)
    for e in entries:
        e["source_thread"] = thread_id  # track which thread it came from
    all_results.extend(entries)

print(f"\nTotal entries: {len(all_results)}")
print(json.dumps(all_results, indent=2))

Processing thread 136...
Processing thread 138...
Processing thread 162...
Processing thread 203...
Processing thread 245...
Processing thread 346...
Processing thread 589...
Processing thread 773...
Processing thread 1168...
Processing thread 1272...
Processing thread 1436...
Processing thread 1459...
Processing thread 1522...
Processing thread 1724...
Processing thread 1905...
Processing thread 1935...
Processing thread 1947...
Processing thread 1976...
Processing thread 2245...
Processing thread 2459...
Processing thread 2656...
Processing thread 2708...
Processing thread 2714...
Processing thread 2718...
Processing thread 2742...
Processing thread 2747...
Processing thread 2840...
Processing thread 2926...
Processing thread 2962...
Processing thread 2992...
Processing thread 3021...
Processing thread 3112...
Processing thread 3209...
Processing thread 3260...
Processing thread 3303...
Processing thread 3418...
Processing thread 3422...
Processing thread 3428...
Processing thread 34

## RAG 7. Save results to a file for review

In [ ]:
# output_file = f"docs/neuron_doc_suggestions.json"

# with open(output_file, "w") as f:
#     json.dump(all_results, f, indent=2)

# print(f"Saved {len(all_results)} entries to {output_file}")

Saved 226 entries to /Users/riesakai/Desktop/MCDOUGAL_LAB/neuron_doc/nrn/docs/neuron_doc_suggestions.json


In [ ]:
# #from tqdm import tqdm
# import openai

# #for idx, row in tqdm(d_lst.iterrows(), total=len(d_lst), desc="Processing rows"):
    
#     converted = prog_ref(row["post"])
#     with open("/Users/riesakai/Desktop/MCDOUGAL_LAB/neuron_forum/neuron_forum_posts/prog_ref1.json", 'a', encoding='utf-8') as f:
#         f.write(f"\n{converted}\n")
#         f.write("-"*100)
#         f.write("\n")

In [ ]:
# def prog_ref(thread, function, category, language):
    
#     client = openai.OpenAI(
#         organization="org-3z6NAgNdNa6W5HskVBdfqbxJ"
#     )

#     prompt = (

#     f"""
#     The post below provides tips for using the {function} {category} in {language} within the NEURON simulator that are not currently included in the official documentation.

#     Please summarize the content in one to two concise sentences to provide additional context for programmers.

#     Return the result in the following JSON format:

#     {{
#     "header": "{function}",
#     "new_doc": "<summary>"
#     }}

#     If relevant, include example code in {language}. Format any code examples using reStructuredText (RST).

#     Post:
#     {thread}
#     """)

#     response = client.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant."},
#             {"role": "user", "content": prompt}
#         ]
#     )

#     converted = response.choices[0].message.content

#     return converted

In [ ]:
# from tqdm import tqdm
# import openai

# for idx, row in tqdm(d_lst.iterrows(), total=len(d_lst), desc="Processing rows"):
    
#     converted = prog_ref(row["post"], row["function"], row["language"], row["function/method/class"])
#     with open("../../neuron_forum/neuron_forum_posts/prog_ref1.json", 'a', encoding='utf-8') as f:
#         f.write(f"\n{converted}\n")
#         f.write("-"*100)
#         f.write("\n")

In [ ]:
# def extract_functions_first_paragraph(path):
#     blocks = []
#     current = []
#     in_para = False

#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             if line.startswith(".. "):
#                 if current:
#                     blocks.append("".join(current))

#                 current = [line]
#                 in_para = False

#             elif current:
#                 if line.strip() == "" and in_para:
#                     blocks.append("".join(current))
#                     current = []
#                 else:
#                     if line.strip():
#                         in_para = True
#                     current.append(line)

#         if current:
#             blocks.append("".join(current))

#     return blocks

In [ ]:
# import re
# import glob

# SPECIFIC_DIRECTIVES = {"function", "class", "method"}

# DIRECTIVE_RE = re.compile(r'^\s*\.\.\s+(\w[\w\-]*)\s*::')
# ALL_CAPS_RE = re.compile(r'^[A-Z][A-Z\s\d_]{1,}$')

# def is_chunk_start(line):
#     stripped = line.rstrip()
#     if not stripped.strip():
#         return False
#     m = DIRECTIVE_RE.match(stripped)
#     if m and m.group(1).lower() in SPECIFIC_DIRECTIVES:
#         return True
#     if ALL_CAPS_RE.match(stripped.strip()):
#         return True
#     return False

# def chunk_rst_text(text, source=""):
#     lines = text.splitlines(keepends=True)
#     chunks = []
#     current_start = 0
#     current_lines = []
#     current_header = None

#     def save_chunk(start, header, lines_buf):
#         content = "".join(lines_buf).strip()
#         if content:
#             chunks.append({
#                 "source": source,
#                 "start_line": start + 1,
#                 "header": header,
#                 "content": content,
#             })

#     for i, line in enumerate(lines):
#         if is_chunk_start(line):
#             if current_lines:
#                 save_chunk(current_start, current_header, current_lines)
#             current_start = i
#             current_header = line.rstrip()
#             current_lines = [line]
#         else:
#             current_lines.append(line)

#     if current_lines:
#         save_chunk(current_start, current_header, current_lines)

#     return chunks

# def chunk_rst_file(filepath):
#     from pathlib import Path
#     text = Path(filepath).read_text(encoding="utf-8", errors="replace")
#     return chunk_rst_text(text, source=filepath)

In [ ]:
# files1 = glob.glob("docs/progref/**/*.rst", recursive=True)
# files2 = glob.glob("docs/nmodl/**/*.rst", recursive=True)

# all_chunks = []
# for f in files1 + files2:
#     all_chunks.extend(chunk_rst_file(f))

# print(f"Total chunks: {len(all_chunks)}")

In [ ]:
# import json
# import openai

# client = openai.OpenAI(
#     organization="org-3z6NAgNdNa6W5HskVBdfqbxJ"
# )
# # Load your supplemental JSON file
# with open("/Users/riesakai/Desktop/MCDOUGAL_LAB/neuron_forum/neuron_forum_posts/prog_ref1.json") as f:
#     supplemental = json.load(f)  # list of {"header": ..., "new_doc": ...}

# def find_matching_chunk(header, chunks):
#     """Find the best matching chunk by header (case-insensitive partial match)."""
#     header_lower = header.lower()
#     for chunk in chunks:
#         if chunk["header"] and header_lower in chunk["header"].lower():
#             return chunk
#     return None

# def integrate_with_gpt(chunk_content, new_doc):
#     response = client.chat.completions.create(
#         model="gpt-4o",
#         messages=[
#             {
#                 "role": "system",
#                 "content": (
#                     "You are a technical documentation editor. "
#                     "You will be given an existing documentation chunk and new supplemental content. "
#                     "Seamlessly integrate the new content into the existing chunk, preserving RST formatting. "
#                     "Do not add any commentary, just return the updated documentation."
#                 )
#             },
#             {
#                 "role": "user",
#                 "content": (
#                     f"Existing documentation:\n\n{chunk_content}\n\n"
#                     f"New supplemental content to integrate:\n\n{new_doc}"
#                 )
#             }
#         ]
#     )
#     return response.choices[0].message.content

# # Process each supplemental entry
# for entry in supplemental:
#     header = entry["header"]
#     new_doc = entry["new_doc"]

#     chunk = find_matching_chunk(header, all_chunks)
#     if chunk:
#         print(f"Matching chunk found for '{header}' — integrating...")
#         updated_content = integrate_with_gpt(chunk["content"], new_doc)
#         chunk["content"] = updated_content  # update in place
#     else:
#         print(f"No matching chunk found for '{header}' — skipping.")

# print("Done! all_chunks is now updated.")

In [ ]:
# for chunk in all_chunks:
#     if chunk["header"] and "connect" in chunk["header"].lower():
#         print(chunk["content"])

In [ ]:
# old prompt

# f"""
# The post below explains tips for using the {function} {category} in {language} in NEURON simulator that are currently not included in the NEURON documentation.
# Please summarize it in one to two sentences to serve as additional information to the programmer's reference in NEURON.
# Begin with the appropriate reStructuredText directive that specifies the relevant function, method, or class (for example: .. class:: IClamp).
# If including example code seems useful, please provide the relevant {language} code as well. Indent the code properly and use the appropriate language-specific directive (for example, .. code-block:: python or .. code-block:: hoc) to format the snippet.
# Return only the final result. Each result must begin with the appropriate directive. Do not include any headers.    

# Post: {thread}

# """

In [ ]:
# import re

# set15 = set(ids_15)
# set14 = set(ids_14)
# pattern = re.compile(r"^TOPIC_ID:\s*(\d+)", re.IGNORECASE | re.MULTILINE)

# posts_in_15 = []
# posts_in_14 = []
# posts_in_both = []

# for post in new_post_chunks:
#     m = pattern.search(post)
#     if not m:
#         continue
#     tid = int(m.group(1))
#     if tid in set15:
#         posts_in_15.append(post)
#     if tid in set14:
#         posts_in_14.append(post)
#     if tid in set15 and tid in set14:
#         posts_in_both.append(post)

In [ ]:
# # find common numbers between (ids_15 or ids_14) and topic_ids
# selected = set(ids_15) & set(ids_14)
# common_nums = sorted(selected & set(topic_ids))

# print(f"Found {len(common_nums)} common ids:")
# print(common_nums)

# # keep variable for further use
# common_nums